# Phase 2: Data Cleaning & Feature Engineering

Mục tiêu của phase này:
- Làm sạch dữ liệu dựa trên kết quả của Phase 1.
- Chuẩn hóa tên cột.
- Tạo surrogate key `order_id`.
- Xử lý missing values và giá trị ngoại lệ (`not_defined`).
- Tạo các feature mới phục vụ phân tích.
- Xuất dữ liệu sạch ra file mới.

In [ ]:
import pandas as pd
import numpy as np

# 1. Load raw data
df = pd.read_csv('../E-commerce Dataset.csv')
print(f"Raw data shape: {df.shape}")
df.head()

## 2. Chuẩn hóa tên cột (Standardize Column Names)
Chuyển tất cả tên cột về định dạng `snake_case`.

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
print("Các cột sau khi chuẩn hóa:")
print(df.columns.tolist())

## 3. Tạo Surrogate Key (`order_id`)
Do dữ liệu gốc không có mã đơn hàng, ta sẽ tạo một `order_id` duy nhất cho mỗi dòng.

In [ ]:
df.insert(0, 'order_id', ['ORD_' + str(i).zfill(6) for i in range(1, len(df) + 1)])
df[['order_id', 'order_date', 'customer_id']].head()

## 4. Xử lý Missing Values & Data Types
Từ báo cáo Phase 1, ta thực hiện các bước làm sạch sau:
- `order_priority`: Điền 'Unknown' cho 2 giá trị thiếu.
- `payment_method`: Đổi 'not_defined' thành 'Unknown', chuẩn hóa format hiển thị.
- `order_date`: Ép kiểu về datetime.
- Các cột số: Ép kiểu numeric, điền missing bằng giá trị trung vị (median) của nhóm `product_category` tương ứng.

In [ ]:
# 4.1 Xử lý Categorical missing/outlier
df['order_priority'] = df['order_priority'].fillna('Unknown')

df['payment_method'] = df['payment_method'].replace('not_defined', 'Unknown')
df['payment_method'] = df['payment_method'].str.replace('_', ' ').str.title()

# 4.2 Convert date
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', errors='coerce')

# 4.3 Xử lý Numeric missing
numeric_cols = ['sales', 'quantity', 'discount', 'profit', 'shipping_cost', 'aging']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # Điền NA bằng median của nhóm danh mục sản phẩm
    df[col] = df.groupby('product_category')[col].transform(lambda x: x.fillna(x.median()))

print("Số lượng missing values sau xử lý:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 5. Feature Engineering
Tạo các feature phục vụ việc phân tích BI:
- Kích thước thời gian: `order_month`, `order_quarter`, `order_weekday`
- Chỉ số tỷ lệ: `profit_margin`, `shipping_cost_ratio`

In [ ]:
# Time features
df['order_month'] = df['order_date'].dt.to_period('M').astype(str)
df['order_quarter'] = 'Q' + df['order_date'].dt.quarter.astype(str) + '-' + df['order_date'].dt.year.astype(str)
df['order_weekday'] = df['order_date'].dt.day_name()

# Ratio features
df['profit_margin'] = np.where(df['sales'] > 0, df['profit'] / df['sales'], 0).round(4)
df['shipping_cost_ratio'] = np.where(df['sales'] > 0, df['shipping_cost'] / df['sales'], 0).round(4)

df[['sales', 'profit', 'profit_margin', 'shipping_cost', 'shipping_cost_ratio', 'order_month', 'order_quarter']].head()

## 6. Export Clean Data
Kiểm tra chất lượng dữ liệu lần cuối và lưu thành file CSV mới.

In [ ]:
import os

df.info()

os.makedirs('../data', exist_ok=True)
clean_path = '../data/cleaned_ecommerce_dataset.csv'
df.to_csv(clean_path, index=False)

print(f"\nData đã được làm sạch và lưu tại: {clean_path}")
print(f"Final shape: {df.shape}")